# PathFinderShip — Lightning AI Benchmark v1

Bu notebook yalnızca hazırlanmış `pathfinder_benchmark_bundle` klasöründe çalıştırılmalıdır. Hücreleri yukarıdan aşağıya sırayla çalıştırın. Tamamlanan deneyler yeniden çalıştırılmaz; notebook kesilirse aynı hücreyi tekrar başlatabilirsiniz.

Önerilen Studio: NVIDIA L4/A10G 24 GB veya daha güçlü GPU, en az 16 GB RAM ve 40 GB boş disk. T4 16 GB kullanılabilir ancak Flan-T5 ve beam-4 bölümleri daha yavaş olacaktır.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, os, subprocess, sys

def find_bundle(start: Path) -> Path:
    candidates = [start, *start.parents]
    candidates += [p for p in start.glob('**/UPLOAD_MANIFEST.json') if p.is_file()]
    for candidate in candidates:
        root = candidate.parent if candidate.is_file() else candidate
        if (root / 'UPLOAD_MANIFEST.json').exists() and (root / 'benchmarks').exists():
            return root.resolve()
    raise FileNotFoundError('UPLOAD_MANIFEST.json bulunan bundle klasörü bulunamadı.')

BUNDLE_ROOT = find_bundle(Path.cwd())
studio_root = Path('/teamspace/studios/this_studio')
OUTPUT_BASE = (studio_root / 'pathfinder_outputs') if studio_root.exists() else (BUNDLE_ROOT / 'outputs')
OUTPUT_BASE.mkdir(parents=True, exist_ok=True)
active_file = OUTPUT_BASE / 'ACTIVE_RUN_ID.txt'
if active_file.exists():
    RUN_ID = active_file.read_text(encoding='utf-8').strip()
else:
    RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
    active_file.write_text(RUN_ID, encoding='utf-8')
RUN_DIR = OUTPUT_BASE / f'pathfinder_results_{RUN_ID}'
DATA_DIR = BUNDLE_ROOT / 'benchmarks' / 'data'
MODELS_ROOT = BUNDLE_ROOT / 'models'
MANIFEST = BUNDLE_ROOT / 'benchmarks' / 'config' / 'experiments.yaml'
os.environ['PYTHONPATH'] = str(BUNDLE_ROOT) + os.pathsep + os.environ.get('PYTHONPATH', '')
print('BUNDLE_ROOT =', BUNDLE_ROOT)
print('RUN_DIR     =', RUN_DIR)
print('RUN_ID      =', RUN_ID)

## 1. Bağımlılıkları kur
Bu hücre birkaç dakika sürer. Lightning tarafından CUDA uyumlu kurulmuş Torch korunur.

In [ ]:
requirements = BUNDLE_ROOT / 'benchmarks' / 'lightning' / 'requirements-lightning.txt'
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', str(requirements)])
print('Bağımlılıklar kuruldu.')

## 2. Donanım ve yükleme paketini doğrula
Her dosya yerel hazırlık sırasında üretilen SHA-256 değeriyle karşılaştırılır. `Mismatch` varsa benchmarka devam etmeyin.

In [ ]:
import hashlib, platform
try:
    subprocess.run(['nvidia-smi'], check=False)
except FileNotFoundError:
    print('UYARI: nvidia-smi bulunamadı.')
upload_manifest = json.loads((BUNDLE_ROOT / 'UPLOAD_MANIFEST.json').read_text(encoding='utf-8'))
errors = []
for item in upload_manifest['files']:
    path = BUNDLE_ROOT / item['relative_path']
    if not path.exists():
        errors.append((item['relative_path'], 'missing'))
        continue
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    if digest != item['sha256']:
        errors.append((item['relative_path'], 'sha256_mismatch'))
assert not errors, f'Yükleme paketi doğrulanamadı: {errors[:10]}'
print(f"Upload doğrulandı: {upload_manifest['file_count']} dosya")

## 3. Resmî IFEval değerlendiricisini ve temiz public test setlerini hazırla
HotpotQA, eğitim kaynaklarıyla çakıştığı için RAGBench seçiminden bilinçli olarak çıkarılmıştır.

In [ ]:
THIRD_PARTY = OUTPUT_BASE / 'third_party'
GOOGLE_RESEARCH = THIRD_PARTY / 'google-research'
if not (GOOGLE_RESEARCH / 'instruction_following_eval').exists():
    THIRD_PARTY.mkdir(parents=True, exist_ok=True)
    subprocess.check_call(['git', 'clone', '--depth', '1', '--filter=blob:none', '--sparse', 'https://github.com/google-research/google-research.git', str(GOOGLE_RESEARCH)])
    subprocess.check_call(['git', '-C', str(GOOGLE_RESEARCH), 'sparse-checkout', 'set', 'instruction_following_eval'])
if not (DATA_DIR / 'public_dataset_manifest.json').exists():
    subprocess.check_call([
        sys.executable, '-m', 'benchmarks.lightning.prepare_public_data',
        '--fingerprints', str(DATA_DIR / 'training_fingerprints.jsonl'),
        '--output-dir', str(DATA_DIR),
    ], cwd=BUNDLE_ROOT)
print((DATA_DIR / 'public_dataset_manifest.json').read_text(encoding='utf-8'))

## 4. Çalıştırma yardımcısı
Her deney kendi durum dosyasını yazar. Bir model hata alırsa diğerleri çalışmaya devam eder.

In [ ]:
def run_experiment(experiment_id: str, force: bool = False):
    command = [
        sys.executable, '-m', 'benchmarks.run',
        '--manifest', str(MANIFEST),
        '--models-root', str(MODELS_ROOT),
        '--data-dir', str(DATA_DIR),
        '--run-dir', str(RUN_DIR),
        '--experiment', experiment_id,
        '--ifeval-code-root', str(GOOGLE_RESEARCH),
    ]
    if force:
        command.append('--force')
    print('RUN:', experiment_id)
    subprocess.run(command, cwd=BUNDLE_ROOT, check=False)
    status_path = RUN_DIR / 'status' / f'{experiment_id}.json'
    print(status_path.read_text(encoding='utf-8') if status_path.exists() else 'Durum dosyası oluşmadı.')

## 5. MiniLM
CPU üzerinde yaklaşık birkaç dakika sürer.

In [ ]:
run_experiment('minilm_intent_int8')

## 6. Erken Flan-T5 Small/Base deneyleri
Üç benzersiz tam fine-tune model ortak Chat+Command setinde çalışır.

In [ ]:
for experiment_id in [
    'flan_t5_small_early',
    'flan_t5_base_early_batch64',
    'flan_t5_base_early_stopping',
]:
    run_experiment(experiment_id)

## 7. Flan-T5 Large LoRA deney zinciri
Bu bölüm en uzun bölümdür. Her adapter Chat, IFEval ve RAG üzerinde aynı protokolle çalışır. Tam matris GPU'ya bağlı olarak birkaç saat sürebilir.

In [ ]:
for experiment_id in [
    'flan_large_lora_qv_failed',
    'flan_large_lora_rag2_step1320',
    'flan_large_lora_chat12_step1485',
    'flan_large_lora_chat12_step1980',
    'flan_large_lora_first_try',
    'flan_large_lora_second_try',
]:
    run_experiment(experiment_id)

## 8. YOLO11 pretrained entegrasyon benchmarkı
COCO val2017 indirilir ve n/s/m/l/x ile l-ONNX doğrulanır. Bu sonuç özel YOLO eğitimi kanıtı olarak kullanılmaz.

In [ ]:
run_experiment('yolo11_pretrained_integration')

## 9. Sonuçları paketle
Son hücre `RUN_COMPLETE.txt`, tablolar, grafikler, SHA manifesti ve indirilecek ZIP'i üretir.

In [ ]:
ZIP_PATH = OUTPUT_BASE / f'{RUN_DIR.name}.zip'
subprocess.check_call([
    sys.executable, '-m', 'benchmarks.report',
    '--run-dir', str(RUN_DIR),
    '--zip-path', str(ZIP_PATH),
    '--historical-evidence', str(BUNDLE_ROOT / 'historical_evidence' / 'historical_extraction.json'),
], cwd=BUNDLE_ROOT)
print('TAMAMLANDI')
print('İndirilecek ZIP:', ZIP_PATH)
print('Bu ZIP dosyasını Windows üzerinde lightning_results_incoming klasörüne koyun.')